TODO:

1. Confirm all E2E scans are exported
2. Load all E2E scans
3. Run preprocessing on each scan
   - extract volume
   - extract RPE/BM
   - flatten using RPE
   - extract below-RPE ROI

4. Compute QC table
   - missing layers
   - failed flattening
   - ROI dimensions
   - intensity summaries/histograms
   - scan quality
   - acquisition parameters

5. Save processed outputs
   - flattened volume or ROI volume
   - metadata/QC CSV

6. Create clinician review file
   - ID
   - representative B-scans to review
   - barcode status: absent/present/unsure
   - optional notes

7. After clinician labeling, merge labels with QC table
   - id
   - barcode status
   - number of B-scans
   - flatten ok
   - ROI extracted
   - dimensions

Final File Structure:
```
id, barcode_status, label_confidence, notes, n_bscans, missing_layers, flatten_ok, roi_extracted, roi_shape, scan_quality
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from barcode.data_loading import (
    find_e2e_files,
    load_e2e,
    print_e2e_summary,
    get_layer_array,
)

DATA_DIR = PROJECT_ROOT / "data" / "heyex"

e2e_files = find_e2e_files(DATA_DIR)
e2e_files

In [ ]:
ev = load_e2e(e2e_files[0])
print_e2e_summary(ev)

In [ ]:
rpe = get_layer_array(ev, "RPE")
bm = get_layer_array(ev, "BM")

print(rpe.shape)
print(bm.shape)

In [ ]:
from barcode.preprocessing import (
    flatten_volume,
    extract_below_layer_roi,
    preprocess_volume_from_layers,
)

rpe = ev.layers["RPE"].data

out = preprocess_volume_from_layers(
    ev.data,
    rpe,
    offset_top=5,
    offset_bottom=160,
    normalization="zscore",
    normalization_mode="global",
)

out["flattened_volume"].shape, out["roi_volume"].shape, out["processed_roi"].shape, out["target_y"]